# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [219]:
class Matrix:
    def __init__(self, n = None, m = None, data = None):
        if data is not None:
            # Validate list of lists
            if not isinstance(data, list) or not isinstance(data[0], list):
                raise ValueError("Data must be a list of lists.")
            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have the same number of columns.")
            self.data = [row[:] for row in data]
            self.n = len(data)
            self.m = len(data[0])
        else:
            # Initialize with zeros using n, m
            if n is None or m is None:
                raise ValueError("Must provide n, m or data.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

## Indexing M[i][j] and M[i, j]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
        else:
            return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value
            
## Matrix Assignment

    def assign(self, other):
        # Assign from another matrix
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError("Matrix sizes do not match. Matrix sizes needs to match for assignment.")
            self.data = [row[:] for row in other.data]

        # Assign from list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("List size does not match matrix size.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    __repr__ = __str__

In [220]:
# Test 1
M1 = Matrix(2, 3)
print("M1 (2x3 zeros):")
print(M1)
print("Shape:", M1.n, M1.m)
print()

# Test 2
data = [[1, 2, 3],
        [4, 5, 6]]
M2 = Matrix(data=data)
print("\nM2 (from data):")
print(M2)
print("Shape:", M2.n, M2.m)

# Test 3
print("\nM2[0][1] =", M2[0][1])   
print("M2[0, 1] =", M2[0, 1])   

# Test 4
M3 = Matrix(2, 3)
print("\nM3 before assign:")
print(M3)
M3.assign(M2)
print("\nM3 after assign(M2):")
print(M3)

# Test 5
new_data = [[7, 8, 9],
            [10, 11, 12]]
M3.assign(new_data)
print("\nM3 after assign(list of lists):")
print(M3)

M1 (2x3 zeros):
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
Shape: 2 3


M2 (from data):
[1, 2, 3]
[4, 5, 6]
Shape: 2 3

M2[0][1] = 2
M2[0, 1] = 2

M3 before assign:
[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]

M3 after assign(M2):
[1, 2, 3]
[4, 5, 6]

M3 after assign(list of lists):
[7, 8, 9]
[10, 11, 12]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [221]:
class Matrix:
    def __init__(self, n = None, m = None, data = None):
        if data is not None:
            # Validate list of lists
            if not isinstance(data, list) or not isinstance(data[0], list):
                raise ValueError("Data must be a list of lists.")
            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have the same number of columns.")
            self.data = [row[:] for row in data]
            self.n = len(data)
            self.m = len(data[0])
        else:
            # Initialize with zeros using n, m
            if n is None or m is None:
                raise ValueError("Must provide n, m or data.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

## __getitem__ modification

    def __getitem__(self, key):
        # M[i, j]
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        # M[i:j] = return a submatrix of rows
        if isinstance(key, slice):
            return Matrix(self.data[key])

        # M[i] = return row i as a list
        if isinstance(key, int):
            return self.data[key]

        raise TypeError("Invalid index type.")

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

## Matrix Assignment

    def assign(self, other):
        # Assign from another matrix
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError("Matrix sizes do not match. Matrix sizes needs to match for assignment.")
            self.data = [row[:] for row in other.data]

        # Assign from list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("List size does not match matrix size.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

## New Code added for Question 2

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        return Matrix(data=[[self.data[j][i] for j in range(self.n)] 
                            for i in range(self.m)])

    def row(self, r):
        return Matrix(data=[self.data[r][:]])

    def column(self, c):
        return Matrix(data=[[self.data[i][c]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        return Matrix(data=[row[m0:m1] for row in self.data[n0:n1]])

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    __repr__ = __str__

In [222]:
# Test matrix
A = Matrix(data=[[1,2,3],
                 [4,5,6],
                 [7,8,9]])

print("A:")
print(A)

# shape()
print("\nshape():", A.shape())

# transpose()
print("\ntranspose():")
print(A.transpose())

# row()
print("\nRow 1 of A:")
print("row(1):")
print(A.row(1))

# column()
print("\nColumn 2 of A:")
print("column(2):")
print(A.column(2))

# to_list()
print("\nto_list():", A.to_list())

# block()
print("\nblock(0,2,1,3):")
print(A.block(0,2,1,3))

# slicing tests
print("\nSplicing tests")
print("A[0:2, 1:3]:")
print(A[0:2, 1:3])

print("\nA[:, 1]:")
print(A[:, 1])

print("\nA[1, :]:")
print(A[1, :])

A:
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]

shape(): (3, 3)

transpose():
[1, 4, 7]
[2, 5, 8]
[3, 6, 9]

Row 1 of A:
row(1):
[4, 5, 6]

Column 2 of A:
column(2):
[3]
[6]
[9]

to_list(): [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

block(0,2,1,3):
[2, 3]
[5, 6]

Splicing tests
A[0:2, 1:3]:
[[4, 5, 6]]

A[:, 1]:
[4, 5, 6]

A[1, :]:
[4, 5, 6]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [223]:
def constant(n, m, c):
    return Matrix(data=[[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    M = zeros(n, n)
    for i in range(n):
        M[i, i] = 1.0
    return M

In [224]:
## Test

C = constant(2, 3, 7)
print("constant(2,3,7):")
print(C)

Z = zeros(3, 2)
print("\nzeros(3,2):")
print(Z)

O = ones(2, 4)
print("\nones(2,4):")
print(O)

I = eye(4)
print("\neye(4):")
print(I)

constant(2,3,7):
[7.0, 7.0, 7.0]
[7.0, 7.0, 7.0]

zeros(3,2):
[0.0, 0.0]
[0.0, 0.0]
[0.0, 0.0]

ones(2,4):
[1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.0, 1.0]

eye(4):
[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [225]:
class Matrix:
    def __init__(self, n = None, m = None, data = None):
        if data is not None:
            # Validate list of lists
            if not isinstance(data, list) or not isinstance(data[0], list):
                raise ValueError("Data must be a list of lists.")
            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have the same number of columns.")
            self.data = [row[:] for row in data]
            self.n = len(data)
            self.m = len(data[0])
        else:
            # Initialize with zeros using n, m
            if n is None or m is None:
                raise ValueError("Must provide n, m or data.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

## __getitem__ modification

    def __getitem__(self, key):
        # M[i, j]
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        # M[i:j] = return a submatrix of rows
        if isinstance(key, slice):
            return Matrix(self.data[key])

        # M[i] = return row i as a list
        if isinstance(key, int):
            return self.data[key]

        raise TypeError("Invalid index type.")

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        # Assign from another matrix
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError("Matrix sizes do not match. Matrix sizes needs to match for assignment.")
            self.data = [row[:] for row in other.data]

        # Assign from list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("List size does not match matrix size.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

## New Code added for Question 2

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        return Matrix(data=[[self.data[j][i] for j in range(self.n)]
                            for i in range(self.m)])

    def row(self, r):
        return Matrix(data=[self.data[r][:]])

    def column(self, c):
        return Matrix(data=[[self.data[i][c]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        return Matrix(data=[row[m0:m1] for row in self.data[n0:n1]])

## New Code for Question 4

    def scalarmul(self, c):
        return Matrix(data=[[c * self.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for addition.")
        return Matrix(data=[[self.data[i][j] + N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for subtraction.")
        return Matrix(data=[[self.data[i][j] - N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError("Inner dimensions must match for matrix multiplication.")
        out = Matrix(self.n, N.m)
        for i in range(self.n):
            for j in range(N.m):
                s = 0
                for k in range(self.m):
                    s += self.data[i][k] * N.data[k][j]
                out[i, j] = s
        return out

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for elementwise multiplication.")
        return Matrix(data=[[self.data[i][j] * N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def equals(self, N):
        if self.shape() != N.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    __repr__ = __str__

In [226]:
## Test

A = Matrix(data=[[1,2,3],
                 [4,5,6]])

B = Matrix(data=[[6,5,4],
                 [3,2,1]])

print("A:")
print(A)

print("\nB:")
print(B)

# Scalar multiplication
print("\nA.scalarmul(2):")
print(A.scalarmul(2))

# Addition
print("\nA.add(B):")
print(A.add(B))

# Subtraction
print("\nA.sub(B):")
print(A.sub(B))

# Matrix multiplication (2x3) * (3x2)
C = Matrix(data=[[1,2],
                 [3,4],
                 [5,6]])

print("\nA.mat_mult(C):")
print(A.mat_mult(C))

# Elementwise multiplication
print("\nA.element_mult(B):")
print(A.element_mult(B))

# Equals
print("\nA.equals(A):", A.equals(A))
print("A.equals(B):", A.equals(B))

A:
[1, 2, 3]
[4, 5, 6]

B:
[6, 5, 4]
[3, 2, 1]

A.scalarmul(2):
[2, 4, 6]
[8, 10, 12]

A.add(B):
[7, 7, 7]
[7, 7, 7]

A.sub(B):
[-5, -3, -1]
[1, 3, 5]

A.mat_mult(C):
[22, 28]
[49, 64]

A.element_mult(B):
[6, 10, 12]
[12, 10, 6]

A.equals(A): True
A.equals(B): False


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [227]:
class Matrix:
    def __init__(self, n = None, m = None, data = None):
        if data is not None:
            # Validate list of lists
            if not isinstance(data, list) or not isinstance(data[0], list):
                raise ValueError("Data must be a list of lists.")
            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have the same number of columns.")
            self.data = [row[:] for row in data]
            self.n = len(data)
            self.m = len(data[0])
        else:
            # Initialize with zeros using n, m
            if n is None or m is None:
                raise ValueError("Must provide n, m or data.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

## __getitem__ modification

    def __getitem__(self, key):
        # M[i, j]
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        # M[i:j] = return a submatrix of rows
        if isinstance(key, slice):
            return Matrix(self.data[key])

        # M[i] = return row i as a list
        if isinstance(key, int):
            return self.data[key]

        raise TypeError("Invalid index type.")

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        # Assign from another matrix
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError("Matrix sizes do not match. Matrix sizes needs to match for assignment.")
            self.data = [row[:] for row in other.data]

        # Assign from list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("List size does not match matrix size.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

## New Code added for Question 2

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        return Matrix(data=[[self.data[j][i] for j in range(self.n)]
                            for i in range(self.m)])

    def row(self, r):
        return Matrix(data=[self.data[r][:]])

    def column(self, c):
        return Matrix(data=[[self.data[i][c]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        return Matrix(data=[row[m0:m1] for row in self.data[n0:n1]])

## New Code for Question 4

    def scalarmul(self, c):
        return Matrix(data=[[c * self.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for addition.")
        return Matrix(data=[[self.data[i][j] + N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for subtraction.")
        return Matrix(data=[[self.data[i][j] - N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError("Inner dimensions must match for matrix multiplication.")
        out = Matrix(self.n, N.m)
        for i in range(self.n):
            for j in range(N.m):
                s = 0
                for k in range(self.m):
                    s += self.data[i][k] * N.data[k][j]
                out[i, j] = s
        return out

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for elementwise multiplication.")
        return Matrix(data=[[self.data[i][j] * N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def equals(self, N):
        if self.shape() != N.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    __repr__ = __str__

## New code for Question 5

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        # M * scalar
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        # M * M (matrix multiplication)
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        raise ValueError("Unsupported operand for *")

    def __rmul__(self, other):
        # scalar * M
        return self.scalarmul(other)

    def __eq__(self, other):
        return self.equals(other)

In [228]:
## Test

A = Matrix(data=[[1,2],
                 [3,4]])

B = Matrix(data=[[5,6],
                 [7,8]])

print("A:")
print(A)

print("\nB:")
print(B)

# M + N
print("\nA + B:")
print(A + B)

# M - N
print("\nA - B:")
print(A - B)

# M * N (matrix multiplication)
print("\nA * B:")
print(A * B)

# M * scalar
print("\nA * 3:")
print(A * 3)

# scalar * M
print("\n3 * A:")
print(3 * A)

# Equal
print("\nA == A:", A == A)
print("A == B:", A == B)

A:
[1, 2]
[3, 4]

B:
[5, 6]
[7, 8]

A + B:
[6, 8]
[10, 12]

A - B:
[-4, -4]
[-4, -4]

A * B:
[19, 22]
[43, 50]

A * 3:
[3, 6]
[9, 12]

3 * A:
[3, 6]
[9, 12]

A == A: True
A == B: False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [229]:
class Matrix:
    def __init__(self, n = None, m = None, data = None):
        if data is not None:
            # Validate list of lists
            if not isinstance(data, list) or not isinstance(data[0], list):
                raise ValueError("Data must be a list of lists.")
            row_len = len(data[0])
            for row in data:
                if len(row) != row_len:
                    raise ValueError("All rows must have the same number of columns.")
            self.data = [row[:] for row in data]
            self.n = len(data)
            self.m = len(data[0])
        else:
            # Initialize with zeros using n, m
            if n is None or m is None:
                raise ValueError("Must provide n, m or data.")
            self.n = n
            self.m = m
            self.data = [[0.0 for _ in range(m)] for _ in range(n)]

## __getitem__ modification

    def __getitem__(self, key):
        # M[i, j]
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]

        # M[i:j] = return a submatrix of rows
        if isinstance(key, slice):
            return Matrix(self.data[key])

        # M[i] = return row i as a list
        if isinstance(key, int):
            return self.data[key]

        raise TypeError("Invalid index type.")

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            self.data[key] = value

    def assign(self, other):
        # Assign from another matrix
        if isinstance(other, Matrix):
            if (self.n, self.m) != (other.n, other.m):
                raise ValueError("Matrix sizes do not match. Matrix sizes needs to match for assignment.")
            self.data = [row[:] for row in other.data]

        # Assign from list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("List size does not match matrix size.")
            self.data = [row[:] for row in other]

        else:
            raise ValueError("Invalid assignment type.")

## New Code added for Question 2

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        return Matrix(data=[[self.data[j][i] for j in range(self.n)]
                            for i in range(self.m)])

    def row(self, r):
        return Matrix(data=[self.data[r][:]])

    def column(self, c):
        return Matrix(data=[[self.data[i][c]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        return Matrix(data=[row[m0:m1] for row in self.data[n0:n1]])

## New Code for Question 4

    def scalarmul(self, c):
        return Matrix(data=[[c * self.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for addition.")
        return Matrix(data=[[self.data[i][j] + N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for subtraction.")
        return Matrix(data=[[self.data[i][j] - N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n:
            raise ValueError("Inner dimensions must match for matrix multiplication.")
        out = Matrix(self.n, N.m)
        for i in range(self.n):
            for j in range(N.m):
                s = 0
                for k in range(self.m):
                    s += self.data[i][k] * N.data[k][j]
                out[i, j] = s
        return out

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrix sizes must match for elementwise multiplication.")
        return Matrix(data=[[self.data[i][j] * N.data[i][j] for j in range(self.m)]
                            for i in range(self.n)])

    def equals(self, N):
        if self.shape() != N.shape():
            return False
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != N.data[i][j]:
                    return False
        return True

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    __repr__ = __str__

## New code for Question 5

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        # M * scalar
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        # M * M (matrix multiplication)
        if isinstance(other, Matrix):
            return self.mat_mult(other)
        raise ValueError("Unsupported operand for *")

    def __rmul__(self, other):
        # scalar * M
        return self.scalarmul(other)

    def __eq__(self, other):
        return self.equals(other)

In [230]:
## Test

A = Matrix(data=[[1, 2],
                 [3, 4]])

B = Matrix(data=[[2, 0],
                 [1, 2]])

C = Matrix(data=[[0, 1],
                 [2, 3]])

print("A:")
print(A)
print("\nB:")
print(B)
print("\nC:")
print(C)
print("\n---\n")


# 1. (AB)C = A(BC)

left = (A * B) * C
right = A * (B * C)

print("(AB)C:")
print(left)
print("\nA(BC):")
print(right)
print("\n(AB)C == A(BC):", left == right)
print("\n---\n")


# 2. A(B + C) = AB + AC

left = A * (B + C)
right = (A * B) + (A * C)

print("A(B + C):")
print(left)
print("\nAB + AC:")
print(right)
print("\nA(B + C) == AB + AC:", left == right)
print("\n---\n")


# 3. AB ≠ BA

AB = A * B
BA = B * A

print("AB:")
print(AB)
print("\nBA:")
print(BA)
print("\nAB == BA:", AB == BA)
print("\n---\n")


# 4. AI = A

I = eye(2)
AI = A * I

print("Identity matrix I:")
print(I)
print("\nAI:")
print(AI)
print("\nAI == A:", AI == A)

A:
[1, 2]
[3, 4]

B:
[2, 0]
[1, 2]

C:
[0, 1]
[2, 3]

---

(AB)C:
[8, 16]
[16, 34]

A(BC):
[8, 16]
[16, 34]

(AB)C == A(BC): True

---

A(B + C):
[8, 11]
[18, 23]

AB + AC:
[8, 11]
[18, 23]

A(B + C) == AB + AC: True

---

AB:
[4, 4]
[10, 8]

BA:
[2, 4]
[7, 10]

AB == BA: False

---

Identity matrix I:
[1.0, 0.0]
[0.0, 1.0]

AI:
[1.0, 2.0]
[3.0, 4.0]

AI == A: True
